# SGLang benchmark on a T4

Same experiments as the vLLM runs, on SGLang:

1. **Sweep:** concurrency {1, 2, 4, 8, 16, 32, 64} × input length {128, 512, 2048}, output 128
2. **Prefix caching:** 1024-token shared prefix + 128 unique tokens, radix cache ON vs OFF

**Fair comparison:** requests are sent by the *same* client as the vLLM runs (`vllm bench serve` 0.34.0).
It is installed in a separate virtual environment, so it doesn't conflict with SGLang's dependencies.

Runtime: **Runtime → Change runtime type → T4 GPU**. Use a fresh runtime (not the one vLLM ran in).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install SGLang (in its own venv, Python 3.12)

Colab runs Python 3.13, and some SGLang dependencies (e.g. `outlines_core`) have no prebuilt wheels for it, so `pip install sglang` tries to compile them and fails.
Installing SGLang into a separate Python 3.12 environment avoids that, and also keeps it away from Colab's preinstalled packages.

In [ ]:
!pip install -q uv
!uv venv /content/sglangenv --python 3.12 -q
!uv pip install -q --python /content/sglangenv/bin/python sglang
!/content/sglangenv/bin/python -c "import sglang; print('sglang', sglang.__version__)"

## 3. Install the benchmark client in a separate venv

A second isolated environment at `/content/benchenv`, and we install the **same vLLM version** used for the vLLM runs.
This takes a few minutes. The client only sends requests; it doesn't use the GPU.

In [ ]:
!uv venv /content/benchenv --python 3.12 -q
!uv pip install -q --python /content/benchenv/bin/python "vllm==0.34.0"
!/content/benchenv/bin/python -c "import vllm; print('client vllm', vllm.__version__)"

## 4. Clone the repo

In [ ]:
!git clone https://github.com/m4schen/vllm-serving-bench.git
%cd /content/vllm-serving-bench
!git log --oneline -3

## 5. Start the SGLang server (radix cache ON, the default)

| vLLM flag | SGLang flag |
|---|---|
| `--dtype half` | `--dtype half` (T4 has no bf16) |
| `--max-model-len 4096` | `--context-length 4096` |
| `--gpu-memory-utilization 0.90` | `--mem-fraction-static 0.88` (static memory for weights + KV cache) |

If the server fails to start on the T4 (check the log in step 6), add `--attention-backend triton`, and if needed `--disable-cuda-graph`.

In [ ]:
!nohup /content/sglangenv/bin/python -m sglang.launch_server \
    --model-path Qwen/Qwen2.5-1.5B-Instruct \
    --dtype half --context-length 4096 --mem-fraction-static 0.88 \
    --host 127.0.0.1 --port 30000 \
    > sglang_server.log 2>&1 &

## 6. Wait until the server is ready

In [ ]:
import time, subprocess, requests

def wait_for_server(url="http://localhost:30000/v1/models", log="sglang_server.log", timeout_s=900):
    start = time.time()
    while time.time() - start < timeout_s:
        try:
            if requests.get(url, timeout=5).ok:
                print(f"Server is ready ({time.time() - start:.0f} s)")
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    print(f"Not ready after {timeout_s} s. Last lines of {log}:")
    print(subprocess.run(["tail", "-n", "40", log], capture_output=True, text=True).stdout)
    return False

wait_for_server()

**KV cache capacity.** SGLang reports it as `max_total_num_tokens`. Compare it with vLLM's **335,872 tokens**.
If the two differ a lot, mention it when comparing results.

In [ ]:
!grep -iE "max_total_num_tokens|KV Cache is allocated|error" sglang_server.log | tail -5
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 7. Send one request by hand

In [ ]:
!curl -s http://localhost:30000/v1/completions \
    -H "Content-Type: application/json" \
    -d '{"model": "Qwen/Qwen2.5-1.5B-Instruct", "prompt": "Explain the KV cache in one sentence:", "max_tokens": 50}'

## 8. Sweep (21 runs, roughly 30–40 minutes)

Same `sweep.sh` as for vLLM, with four settings changed:
- `BACKEND=openai`: send standard OpenAI completions requests (SGLang is OpenAI-compatible)
- `BASE_URL`: SGLang listens on port 30000
- `TAG=t4_sglang`: results go to `results/t4_sglang/`
- `BENCH_CMD`: use the client from the separate venv

Seeds are the same as in the vLLM sweep, so both servers get identical prompts. If Colab disconnects, rerun this cell: finished runs are skipped.

In [ ]:
!BACKEND=openai BASE_URL=http://localhost:30000 TAG=t4_sglang \
  BENCH_CMD="/content/benchenv/bin/vllm bench serve" bash sweep.sh

## 9. Check the results are valid

Every run should have **0 failed requests** and exactly **128 output tokens per request**.
If output tokens are short, SGLang ignored `ignore_eos`, and the runs aren't comparable with vLLM.

In [ ]:
import json, glob, re

rows = []
for f in glob.glob("results/t4_sglang/*.json"):
    d = json.load(open(f))
    c, i = map(int, re.search(r"c(\d+)_in(\d+)", f).groups())
    ok = d["failed"] == 0 and d["total_output_tokens"] == d["completed"] * 128
    rows.append((i, c, d["output_throughput"], d["median_tpot_ms"], d["p99_ttft_ms"], "OK" if ok else "CHECK"))

print(f"{'in':>5} {'c':>3} {'tok/s':>8} {'p50TPOT':>8} {'p99TTFT':>8}  valid")
for i, c, thr, tpot, ttft, ok in sorted(rows):
    print(f"{i:5d} {c:3d} {thr:8.1f} {tpot:8.1f} {ttft:8.0f}  {ok}")

## 10. Prefix caching ON (radix cache on)

Same requests as the vLLM prefix experiment: a 1024-token shared prefix + 128 unique tokens, 128 output tokens, same seeds.
`Total input tokens` should be (number of requests) × 1152.

In [ ]:
%%bash
BENCH="/content/benchenv/bin/vllm bench serve"
for c in 1 16 64; do
  n=$(( c * 5 )); (( n < 32 )) && n=32
  echo "=== SGLang prefix ON, concurrency $c ==="
  $BENCH --backend openai --base-url http://localhost:30000 --model Qwen/Qwen2.5-1.5B-Instruct \
    --dataset-name random \
    --random-prefix-len 1024 --random-input-len 128 --random-output-len 128 \
    --ignore-eos --num-prompts $n --max-concurrency $c \
    --percentile-metrics ttft,tpot,itl,e2el --metric-percentiles 50,99 \
    --seed $(( 200 + c )) \
    --save-result --result-dir results/t4_sglang_prefix_on --result-filename c${c}.json \
    2>&1 | grep -E "Total input tokens|Total generated tokens|Output token throughput|Mean TTFT|P99 TTFT|Mean TPOT"
done

## 11. Restart with the radix cache OFF

`[s]glang` is a pattern trick: it matches "sglang" in other processes, but not this shell's own command line, so the wait loop can't wait on itself.

In [ ]:
!pkill -f "[s]glang.launch_server"
!while pgrep -f "[s]glang" > /dev/null; do sleep 2; done; echo "Old server stopped"
!nohup /content/sglangenv/bin/python -m sglang.launch_server \
    --model-path Qwen/Qwen2.5-1.5B-Instruct \
    --dtype half --context-length 4096 --mem-fraction-static 0.88 \
    --host 127.0.0.1 --port 30000 \
    --disable-radix-cache \
    > sglang_server_nocache.log 2>&1 &

In [ ]:
wait_for_server(log="sglang_server_nocache.log")
!grep -oE "disable_radix_cache=[A-Za-z]+" sglang_server_nocache.log | head -1
!pgrep -af "[s]glang" | head

The `grep` should print `disable_radix_cache=True`, and `pgrep` should show only one server.

## 12. Prefix caching OFF (same seeds, same requests)

In [ ]:
%%bash
BENCH="/content/benchenv/bin/vllm bench serve"
for c in 1 16 64; do
  n=$(( c * 5 )); (( n < 32 )) && n=32
  echo "=== SGLang prefix OFF, concurrency $c ==="
  $BENCH --backend openai --base-url http://localhost:30000 --model Qwen/Qwen2.5-1.5B-Instruct \
    --dataset-name random \
    --random-prefix-len 1024 --random-input-len 128 --random-output-len 128 \
    --ignore-eos --num-prompts $n --max-concurrency $c \
    --percentile-metrics ttft,tpot,itl,e2el --metric-percentiles 50,99 \
    --seed $(( 200 + c )) \
    --save-result --result-dir results/t4_sglang_prefix_off --result-filename c${c}.json \
    2>&1 | grep -E "Total input tokens|Total generated tokens|Output token throughput|Mean TTFT|P99 TTFT|Mean TPOT"
done

## 13. Record versions and download the results

In [ ]:
!/content/sglangenv/bin/python -c "import sglang; print('server: sglang', sglang.__version__)"
!/content/benchenv/bin/python -c "import vllm; print('client: vllm', vllm.__version__)"
!grep -iE "max_total_num_tokens" sglang_server.log | tail -1
!zip -qr results_sglang.zip results/t4_sglang results/t4_sglang_prefix_on results/t4_sglang_prefix_off
from google.colab import files
files.download("results_sglang.zip")